# Φάση Γ: Classification Models

**Υπεύθυνος:** ML Engineer

**Μοντέλα:**
1. Random Forest
2. SVM


In [6]:
# Φάση Γ: Classification Models
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType
from pyspark.ml.classification import RandomForestClassifier, LinearSVC, NaiveBayes, MultilayerPerceptronClassifier

print("Εκκίνηση SparkSession...")
spark = SparkSession.builder.appName("Stroke_Classification").master("local[*]").getOrCreate()

print("Φόρτωση δεδομένων...")
train_gold = spark.read.parquet("../data/train_gold.parquet")
test_gold = spark.read.parquet("../data/test_gold.parquet")

train_gold = train_gold.withColumn("stroke", train_gold["stroke"].cast(DoubleType()))
test_gold = test_gold.withColumn("stroke", test_gold["stroke"].cast(DoubleType()))

# 1. Random Forest
print("Εκπαίδευση Random Forest...")
rf = RandomForestClassifier(featuresCol="features", labelCol="stroke", numTrees=100, seed=42)
rf_preds = rf.fit(train_gold).transform(test_gold)

# 2. Support Vector Machine (SVM)
print("Εκπαίδευση SVM...")
svm = LinearSVC(featuresCol="features", labelCol="stroke", maxIter=100)
svm_preds = svm.fit(train_gold).transform(test_gold)

# 3. Naive Bayes
print("Εκπαίδευση Naive Bayes...")
nb = NaiveBayes(featuresCol="features", labelCol="stroke")
nb_preds = nb.fit(train_gold).transform(test_gold)

# 4. Neural Network (MLP)
print("Εκπαίδευση Neural Network (MLP)...")
input_size = len(train_gold.select("features").first()[0])
mlp = MultilayerPerceptronClassifier(maxIter=100, layers=[input_size, 16, 8, 2], seed=42, featuresCol="features", labelCol="stroke")
mlp_preds = mlp.fit(train_gold).transform(test_gold)

# Αποθήκευση
print("Αποθήκευση προβλέψεων...")
rf_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/rf_predictions.parquet")
svm_preds.select("stroke", "prediction", "rawPrediction").write.mode("overwrite").parquet("../data/svm_predictions.parquet")
nb_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/nb_predictions.parquet")
mlp_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/mlp_predictions.parquet")

spark.stop()
print("Ολοκληρώθηκε η Φάση Γ!")

Εκκίνηση SparkSession...
Φόρτωση δεδομένων...
Εκπαίδευση Random Forest...
Εκπαίδευση SVM...
Εκπαίδευση Naive Bayes...
Εκπαίδευση Neural Network (MLP)...
Αποθήκευση προβλέψεων...


26/06/03 19:54:19 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1526)
	at org.apache.spark.executor.Executor.$anonfun$heartbeater$1(Executor.scala:483)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18)
	at org.apache.spark.util.Utils$.logUncaughtExceptions(Utils.scala:1941)
	at org.apache.spark.Heartbeater$$anon$1.run(Heartbeater.scala:46)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:572)
	at java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:358)
	at java.base/

Ολοκληρώθηκε η Φάση Γ!
